# Chapter 16 — Read Direct and pump-off HB

TARGET API · CONVERGING · not executable on the current runtime

> **TARGET API / CONVERGING — not executable on the current runtime.**
> These retained numerical request/result cells are not execution
> evidence.

One complete physical Plan supports two independently declared questions
here: Direct response and pump-off HB response on the same selected
View. Their grids remain independent, so this Chapter displays their
complex results without interpolation or a pointwise comparison claim.

## Lesson 16.1 — Build the feedline

### Declare the root Plan and N=1 feedline

This fresh-kernel lesson repeats the same three root Subsystems:
`feedline`, `readout`, and `floating`. The CPW bodies are finite-pi
discretizations, not exact distributed equivalents.

In [ ]:
from scnsim import CircuitPlan, RLGC, components, units as u

plan = CircuitPlan(id="floating_probe_course")
feedline = plan.subsystem(id="feedline")
rlgc = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0]] * u.pF / u.m,
)

`plan`, `feedline`, and `rlgc` define the common physical starting
point. The following declarations create its two line bodies, then their
same-net taps.

In [ ]:
left = feedline.add(
    components.transmission_line(
        id="left",
        length=1.0 * u.mm,
        rlgc=rlgc,
        n_sections=1,
    )
)
right = feedline.add(
    components.transmission_line(
        id="right",
        length=1.0 * u.mm,
        rlgc=rlgc,
        n_sections=1,
    )
)

`left` and `right` are independent N=1 bodies. Their oriented signal
pins are named in the next cell before the three public feedline handles
are exposed.

In [ ]:
input_bus = feedline.bus(id="input")
middle_bus = feedline.bus(id="middle")
output_bus = feedline.bus(id="output")

`middle_bus` is the shared electrical endpoint of the two line sections
and the public coupling pin. The next cell consumes it in both series
relations.

In [ ]:
left_head_pin = left.pin("head", conductor="signal")
left_tail_pin = left.pin("tail", conductor="signal")
right_head_pin = right.pin("head", conductor="signal")
right_tail_pin = right.pin("tail", conductor="signal")

left_section = feedline.series(
    id="left_section",
    start=input_bus,
    elements=(
        left.between(left_head_pin, left_tail_pin),
    ),
    end=middle_bus,
)
right_section = feedline.series(
    id="right_section",
    start=middle_bus,
    elements=(
        right.between(right_head_pin, right_tail_pin),
    ),
    end=output_bus,
)
feedline_input_pin = feedline.expose_pin(id="input", at=input_bus)
feedline_coupling_pin = feedline.expose_pin(id="tap", at=middle_bus)
feedline_output_pin = feedline.expose_pin(id="output", at=output_bus)

The CPW reference conductor is metadata only: no reference pin is
exposed or grounded. The resulting `feedline_input_pin`,
`feedline_coupling_pin`, and `feedline_output_pin` values are the only
feedline PinRefs used by the root.

## Lesson 16.2 — Add the grounded readout

### Declare the grounded readout LC

The 110 fF/5.8 nH grounded readout LC is the local lumped approximation
for the target quarter-wave mode. It is not an exact
distributed-equivalence claim.

In [ ]:
readout = plan.subsystem(id="readout")
readout_capacitor = readout.add(
    components.capacitor(
        id="capacitor",
        capacitance=110.0 * u.fF,
    )
)
readout_inductor = readout.add(
    components.inductor(
        id="inductor",
        inductance=5.8 * u.nH,
    )
)
readout_bus = readout.bus(id="node")
readout_parallel = readout.parallel(
    id="parallel_lc",
    start=readout_bus,
    branches=((readout_capacitor,), (readout_inductor,)),
    end=readout.ground,
)
readout_terminal = readout.expose_pin(
    id="readout_node",
    at=readout_bus,
)

`readout_terminal` publishes the fixed grounded-LC boundary that the
root assembly consumes next.

## Lesson 16.3 — Add the floating subsystem

### Declare the fixed floating pair

In [ ]:
floating = plan.subsystem(id="floating")
plus_bus = floating.bus(id="plus")
minus_bus = floating.bus(id="minus")

mutual_cap = floating.add(
    components.capacitor(id="mutual_cap", capacitance=16.0 * u.fF)
)
mutual_ind = floating.add(
    components.inductor(id="mutual_ind", inductance=7.0 * u.nH)
)
plus_shunt = floating.add(
    components.capacitor(
        id="plus_shunt",
        capacitance=45.0 * u.fF,
    )
)
minus_shunt = floating.add(
    components.capacitor(
        id="minus_shunt",
        capacitance=42.0 * u.fF,
    )
)

Those fixed native leaves feed the next cell, which turns them into the
pair structure and returns its two public pins.

In [ ]:
mutual_network = floating.parallel(
    id="mutual_network",
    start=plus_bus,
    branches=((mutual_cap,), (mutual_ind,)),
    end=minus_bus,
)
plus_branch = floating.branch(
    id="plus_shunt",
    at=plus_bus,
    elements=(plus_shunt,),
    end=floating.ground,
)
minus_branch = floating.branch(
    id="minus_shunt",
    at=minus_bus,
    elements=(minus_shunt,),
    end=floating.ground,
)
floating_plus_pin = floating.expose_pin(id="floating_plus", at=plus_bus)
floating_minus_pin = floating.expose_pin(id="floating_minus", at=minus_bus)

`floating_plus_pin` and `floating_minus_pin` are the child boundary. The
root cell next links them and creates the coordinate aliases used by the
shared View.

## Lesson 16.4 — Assemble root couplers and probes

### Declare root coordinate buses and link child boundaries

`floating_plus` and `floating_minus` are root `ElectricNodeRef` aliases.
They are never child pins or private floating leaves.

In [ ]:
feedline_in_bus = plan.bus(id="feedline_in")
feedline_out_bus = plan.bus(id="feedline_out")
readout_root_bus = plan.bus(id="readout_node")
floating_plus_bus = plan.bus(id="floating_plus")
floating_minus_bus = plan.bus(id="floating_minus")

feedline_in_node = feedline_in_bus.node
feedline_out_node = feedline_out_bus.node
readout_node = readout_root_bus.node
floating_plus = floating_plus_bus.node
floating_minus = floating_minus_bus.node

plan.link(
    id="feedline_input_child",
    endpoints=(feedline_in_bus, feedline_input_pin),
)
plan.link(
    id="feedline_output_child",
    endpoints=(feedline_out_bus, feedline_output_pin),
)
plan.link(id="readout_child", endpoints=(readout_root_bus, readout_terminal))
plan.link(
    id="floating_plus_child",
    endpoints=(floating_plus_bus, floating_plus_pin),
)
plan.link(
    id="floating_minus_child",
    endpoints=(floating_minus_bus, floating_minus_pin),
)

The root aliases and public child links are complete. Register native
couplers next, then place their direct handles in the three semantic
series relations.

### Register the three native root couplers

In [ ]:
feedline_coupler = plan.add(
    components.capacitor(
        id="feedline_readout_coupler",
        capacitance=6.0 * u.fF,
    )
)
plus_coupler = plan.add(
    components.capacitor(
        id="readout_to_floating_plus",
        capacitance=4.0 * u.fF,
    )
)
minus_coupler = plan.add(
    components.capacitor(
        id="readout_to_floating_minus",
        capacitance=3.0 * u.fF,
    )
)

Each direct coupler is now ready for exactly one ordered root
connection.

### Place the three couplers in semantic series order

In [ ]:
feedline_readout = plan.series(
    id="feedline_readout",
    start=feedline_coupling_pin,
    elements=(feedline_coupler,),
    end=readout_root_bus,
)
readout_floating_plus = plan.series(
    id="readout_floating_plus",
    start=readout_root_bus,
    elements=(plus_coupler,),
    end=floating_plus_bus,
)
readout_floating_minus = plan.series(
    id="readout_floating_minus",
    start=readout_root_bus,
    elements=(minus_coupler,),
    end=floating_minus_bus,
)

The physical root now exposes `floating_plus` and `floating_minus` as
`ElectricNodeRef` aliases. The next cell adds the raw loads whose
identity the selected View preserves or compensates explicitly.

### Promote terminated Ports and raw probe loads

In [ ]:
feedline_in_port = plan.add_port(
    id="feedline_in",
    at=feedline_in_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
feedline_out_port = plan.add_port(
    id="feedline_out",
    at=feedline_out_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
probe_plus = plan.add_port(
    id="floating_probe_plus",
    at=floating_plus_bus,
    role="nonloading_probe",
    reference_impedance=50.0 * u.ohm,
)
probe_minus = plan.add_port(
    id="floating_probe_minus",
    at=floating_minus_bus,
    role="nonloading_probe",
    reference_impedance=50.0 * u.ohm,
)

`feedline_in_port`, `feedline_out_port`, `probe_plus`, and `probe_minus`
are the raw handles consumed by the common selected View in the next
cell.

### Build the common selected View

In [ ]:
from scnsim import CircuitRun, ReductionPipeline

run = CircuitRun(plan=plan, workspace="workspaces/advanced-course")
raw_loaded_view = run.original
comparison_pipeline = ReductionPipeline().ptc(
    probe_plus,
    probe_minus,
).transform_pair(
    floating_plus,
    floating_minus,
    id="floating",
).retain(
    "feedline_in",
    "feedline_out",
    "floating.differential",
)
view = raw_loaded_view.reduce(comparison_pipeline)

`view` is shared intentionally, while the two request grids remain
independent. The next cell creates both specs without interpolating or
aligning either grid.

## Lesson 16.5 — Read independent Direct and HB results

### Construct independent Direct and HB requests

Direct and HB share this View but retain independent grid declarations.
Their complex S/Y/Z surfaces share the selected-network convention;
magnitude in a plot is presentation only, never a replacement for
complex S.

In [ ]:
from scnsim import (
    CurrentDrive,
    DirectSolveSpec,
    HBCaseSpec,
    HBSolveSpec,
    HBTruncation,
    PumpAxis,
    SParameterTrace,
)

direct_frequencies = [5.5, 6.0, 6.5] * u.GHz
direct_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(),
    output_port="feedline_out",
    output_mode=(),
)
direct_spec = DirectSolveSpec(
    frequencies=direct_frequencies,
    traces=(direct_trace,),
)

pump = PumpAxis(id="pump", frequency=9.0 * u.GHz)
pump_drive = CurrentDrive(
    id="pump_drive",
    at=feedline_in_port,
    mode=(1,),
)
hb_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(0,),
    output_port="feedline_out",
    output_mode=(0,),
)
hb_spec = HBSolveSpec(
    pump_axes=(pump,),
    drives=(pump_drive,),
    frequencies=[5.4, 5.9, 6.4] * u.GHz,
    cases=(HBCaseSpec(id="pump_off", currents={}),),
    truncation=HBTruncation(
        pump_harmonics=(3,),
        modulation_harmonics=(1,),
        three_wave_mixing=False,
        four_wave_mixing=True,
    ),
    traces=(hb_trace,),
)

`direct_spec` and `hb_spec` retain their own frequency lists and typed
outputs. The solve cell consumes the common `view` with each request
separately.

### Solve the two independently declared requests

In [ ]:
direct = run.solve(view, direct_spec)
hb = run.solve(view, hb_spec)

`direct` and `hb` are separate result objects. The final cell displays
the Direct full-complex surfaces and guards every HB case-specific
access.

### Inspect Direct and guarded HB surfaces

There is no interpolation, pointwise grid alignment, peak matching, or
comparison residual in this lesson. A valid failed HB case is not a
trace.

In [ ]:
from IPython.display import display

direct.traces["transmission"].show(magnitude="db")
display(direct.s.view)
display(direct.y.view)
display(direct.z.view)

pump_off = hb.cases["pump_off"]
if pump_off.succeeded:
    pump_off.traces["transmission"].show(magnitude="db")
    display(pump_off.s.view)
    display(pump_off.y.view)
    display(pump_off.z.view)
    display(pump_off.states)
    display(pump_off.state_node_map)
else:
    display(pump_off.failure)

The two curves may share a figure only as separate, identity-bearing
traces. Any numerical comparison belongs to an explicit later
calculation on compatible materialized quantities.

[Previous](15_prepare_hb.qmd) · [Course map](../../docs/index.qmd)